# Dataset (FaceForensics++) Preprocessing

### Importing Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## Part 1 - Data Preprocessing

In [ ]:
# Downloading the Data
import kagglehub

path = kagglehub.dataset_download("hungle3401/faceforensics")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'faceforensics' dataset.
Path to dataset files: /kaggle/input/faceforensics


In [ ]:
# Split Videos
import os
import shutil
import random

input_root = "/kaggle/input/faceforensics/FF++"
output_root = "/content/FF++_split"

classes = ["fake", "real"]

# split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

for cls in classes:
    videos = os.listdir(os.path.join(input_root, cls))
    videos = [v for v in videos if v.endswith((".mp4", ".avi", ".mov", ".mkv"))]

    random.shuffle(videos)

    total = len(videos)
    train_end = int(train_ratio * total)
    val_end = int((train_ratio + val_ratio) * total)

    splits = {
        "train": videos[:train_end],
        "val": videos[train_end:val_end],
        "test": videos[val_end:]
    }

    for split_name, split_videos in splits.items():
        split_dir = os.path.join(output_root, split_name, cls)
        os.makedirs(split_dir, exist_ok=True)

        for v in split_videos:
            src = os.path.join(input_root, cls, v)
            dst = os.path.join(split_dir, v)
            shutil.copy(src, dst)

print("✅ Video-level split completed.")

✅ Video-level split completed.


In [ ]:
# Verify
for split in ["train", "val", "test"]:
    for cls in ["fake", "real"]:
        path = f"/content/FF++_split/{split}/{cls}"
        print(split, cls, len(os.listdir(path)))

train fake 140
train real 140
val fake 30
val real 30
test fake 30
test real 30


In [ ]:
train_videos = set(os.listdir("/content/FF++_split/train/fake") +
                   os.listdir("/content/FF++_split/train/real"))

val_videos = set(os.listdir("/content/FF++_split/val/fake") +
                 os.listdir("/content/FF++_split/val/real"))

test_videos = set(os.listdir("/content/FF++_split/test/fake") +
                  os.listdir("/content/FF++_split/test/real"))

print("Train ∩ Val:", len(train_videos & val_videos))
print("Train ∩ Test:", len(train_videos & test_videos))
print("Val ∩ Test:", len(val_videos & test_videos))

Train ∩ Val: 0
Train ∩ Test: 0
Val ∩ Test: 0


In [ ]:
# Extract Face Frames from Videos
import os
import cv2
from tqdm import tqdm

input_root = "/content/FF++_split"
output_root = "/content/faces_split"

splits = ["train", "val", "test"]
classes = ["fake", "real"]

img_size = (224, 224)
frame_skip = 10
video_exts = (".mp4", ".avi", ".mov", ".mkv")

face_detector = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

for split in splits:
    for cls in classes:
        input_dir = os.path.join(input_root, split, cls)
        output_dir = os.path.join(output_root, split, cls)
        os.makedirs(output_dir, exist_ok=True)

        video_list = [
            v for v in os.listdir(input_dir)
            if v.lower().endswith(video_exts)
        ]

        # 🔥 tqdm here (video-level progress)
        for video_name in tqdm(video_list, desc=f"{split}/{cls}", unit="video"):
            video_path = os.path.join(input_dir, video_name)
            cap = cv2.VideoCapture(video_path)

            frame_id = 0
            saved_id = 0
            base_name = os.path.splitext(video_name)[0]

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_id % frame_skip == 0:
                    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

                    faces = face_detector.detectMultiScale(
                        gray,
                        scaleFactor=1.1,
                        minNeighbors=5,
                        minSize=(30, 30)
                    )

                    if len(faces) > 0:
                        # largest face
                        faces = sorted(faces, key=lambda x: x[2] * x[3], reverse=True)
                        x, y, w, h = faces[0]

                        face = frame[y:y+h, x:x+w]
                        face = cv2.resize(face, img_size)

                        save_name = f"{base_name}_{saved_id:04d}.jpg"
                        save_path = os.path.join(output_dir, save_name)
                        cv2.imwrite(save_path, face)

                        saved_id += 1

                frame_id += 1

            cap.release()

print("Done extracting face crops.")

train/fake:  72%|███████▏  | 101/140 [1:18:51<31:21, 48.23s/video]

In [ ]:
for split in ["train", "val", "test"]:
    for cls in ["fake", "real"]:
        path = f"/content/faces_split/{split}/{cls}"
        print(split, cls, len(os.listdir(path)))

train fake 10094
train real 11815
val fake 1988
val real 2516
test fake 2233
test real 2664


In [ ]:
# VISUAL CHECK
import matplotlib.pyplot as plt
import os
import cv2
import random

sample_dir = "/content/faces_split/train/fake"
files = random.sample(os.listdir(sample_dir), 5)

plt.figure(figsize=(15,3))
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(sample_dir, f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.axis("off")

plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step B: after processing, copy to Drive
!cp -r /content/faces_split /content/drive/MyDrive/

## Part 2 - Building the CNN

## Part 3 - Training the CNN

## Part 4 - Making a single prediction